In [55]:
import pandas as pd
import numpy as np

In [56]:
def split_into_spans(npy_arr, span_size):
    n = len(npy_arr)
    res = []
    i = 0
    while i < n:
        cur = npy_arr[i : i + span_size]
        res.append(cur)
        i += span_size
    return res

In [75]:
trips = pd.read_csv('bixi-clean.csv')
table = dict()
table['duration_sec'] = trips['duration_sec'].to_numpy()
table['longitude_x'] = trips['longitude_x'].to_numpy()
table['latitude_x'] = trips['latitude_x'].to_numpy()
table['longitude_y'] = trips['longitude_y'].to_numpy()
table['latitude_y'] = trips['latitude_y'].to_numpy()
bixi = {'table': table, 'matrix': None}
print(bixi)

table = bixi['table']
dur = table['duration_sec']
lon_x = table['longitude_x']
lat_x = table['latitude_x']
lon_y = table['longitude_y']
lat_y = table['latitude_y']

num_rows = 3887650
dur = split_into_spans(dur, num_rows // 4)
lon_x = split_into_spans(lon_x, num_rows // 4)
lat_x = split_into_spans(lat_x, num_rows // 4)
lon_y = split_into_spans(lon_y, num_rows // 4)
lat_y = split_into_spans(lat_y, num_rows // 4)

dur = np.concatenate(dur)
lon_x = np.concatenate(lon_x)
lat_x = np.concatenate(lat_x)
lon_y = np.concatenate(lon_y)
lat_y = np.concatenate(lat_y)

print(dur, lon_x, lat_x, lon_y, lat_y, sep='\n')

{'table': {'duration_sec': array([1841,  553,  195, ..., 3363,  179,  306]), 'longitude_x': array([-73.57156895, -73.56950909, -73.57208   , ..., -73.548687  ,
       -73.55199   , -73.584157  ]), 'latitude_x': array([45.46300109, 45.51908844, 45.50781   , ..., 45.55896   ,
       45.52353   , 45.540881  ]), 'longitude_y': array([-73.57156895, -73.56950909, -73.57477158, ..., -73.57547   ,
       -73.55786   , -73.575515  ]), 'latitude_y': array([45.46300109, 45.51908844, 45.5081439 , ..., 45.51059   ,
       45.519157  , 45.546978  ])}, 'matrix': None}
[1841  553  195 ... 3363  179  306]
[-73.57156895 -73.56950909 -73.57208    ... -73.548687   -73.55199
 -73.584157  ]
[45.46300109 45.51908844 45.50781    ... 45.55896    45.52353
 45.540881  ]
[-73.57156895 -73.56950909 -73.57477158 ... -73.57547    -73.55786
 -73.575515  ]
[45.46300109 45.51908844 45.5081439  ... 45.51059    45.519157
 45.546978  ]


In [58]:
def haversine_distance(longitude_x, latitude_x, longitude_y, latitude_y):
    # Convert latitude and longitude from degrees to radians
    longitude_x = np.radians(longitude_x)
    latitude_x = np.radians(latitude_x)
    longitude_y = np.radians(longitude_y)
    latitude_y = np.radians(latitude_y)
    
    # Haversine formula
    dlon = longitude_y - longitude_x
    dlat = latitude_y - latitude_x
    a = np.sin(dlat / 2)**2 + np.cos(latitude_x) * np.cos(latitude_y) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    # Radius of Earth in kilometers (mean radius)
    r = 6371.0
    
    # Distance in kilometers
    distance_km = r * c
    
    # Convert distance to meters
    distance_m = distance_km * 1000
    
    return distance_m

In [59]:
table['distance'] = haversine_distance(lon_x, lat_x, lon_y, lat_y)
dist = table['distance']
print(dist)

[   0.            0.          213.0061901  ... 5768.89148476  667.52220845
  955.28369595]


In [60]:
shuffle_ixs = np.random.permutation(len(dist))
dist = dist[shuffle_ixs]
dur = dur[shuffle_ixs]

In [61]:
max_dist = np.max(dist)
max_dur = np.max(dur)

dist = dist / max_dist
dur = dur / max_dur

In [62]:
train_ratio = 0.7
test_ratio = 0.3
split_ix = int(len(dist) * 0.7)

dist_train = dist[ : split_ix]
dur_train = dur[ : split_ix]

dist_test = dist[split_ix : ]
dur_test = dur[split_ix : ]

In [63]:
ones_train = np.ones((len(dist_train),))
train_in = np.stack((ones_train, dist_train), axis=-1)
train_out = dur_train
params = np.ones((1, 2))

ones_test = np.ones((len(dist_test),))
test_in = np.stack((ones_test, dist_test), axis=-1)
test_out = dur_test

In [64]:
pred = train_in @ params.T
pred = np.reshape(pred, -1)

In [65]:
def squared_err(act, pred):
    errors = np.square(pred - act)
    sum_err = np.sum(errors)
    num_vals = act.shape[0]
    res = sum_err / (2 * num_vals)
    return res

In [66]:
sq_err = squared_err(train_out, pred)
print(sq_err)

0.5218200831757618


In [67]:
def grad_desc(act, pred, indata):
    return (pred - act).T @ indata / act.shape[0]

In [68]:
alpha = 0.1

params = params - alpha * grad_desc(train_out, pred, train_in)
print(params)

[[0.89819307 0.98588475]]


In [69]:
pred = train_in @ params.T
pred = np.reshape(pred, -1)
sq_err = squared_err(train_out, pred)
print(sq_err)

0.4215595567424591


In [70]:
for i in range(100):
    pred = train_in @ params.T
    pred = np.reshape(pred, -1)
    params = params - alpha * grad_desc(train_out, pred, train_in)
    sq_err = squared_err(train_out, pred)
    
    if( (i+1) % 100 == 0):
        print(f"Error rate after {i + 1} iterations is {sq_err}")
    
print(params)
sq_err = squared_err(train_out, pred)
print(sq_err)

Error rate after 100 iterations is 0.00302999482497054
[[0.00355897 0.83675075]]
0.00302999482497054


In [71]:
test_pred = test_in @ params.T
test_pred = np.reshape(test_pred, -1)

sq_err = squared_err(test_out * max_dur, test_pred * max_dur)
print(sq_err)

157118.04174712248
